# hatch_fancy_pypi_readme smoke test

Smoke test for `hatch-fancy-pypi-readme`, a hatchling metadata hook that assembles a package's PyPI
README from text and file fragments plus regex substitutions. Every example runs offline in a
temporary directory; nothing is built or uploaded.

## Example 1: Hatchling plugin registration

In [ ]:
# Example 1: The package is a hatchling metadata hook, registered under the plugin name "fancy-pypi-readme".
from importlib.metadata import entry_points

import hatch_fancy_pypi_readme
from hatch_fancy_pypi_readme.hooks import FancyReadmeMetadataHook
from hatchling.metadata.plugin.interface import MetadataHookInterface

print(f"hook class: {FancyReadmeMetadataHook.__module__}.{FancyReadmeMetadataHook.__qualname__}")
print(f"plugin name: {FancyReadmeMetadataHook.PLUGIN_NAME}")
assert FancyReadmeMetadataHook.PLUGIN_NAME == "fancy-pypi-readme", FancyReadmeMetadataHook.PLUGIN_NAME
assert issubclass(FancyReadmeMetadataHook, MetadataHookInterface)

hatch_plugins = {ep.name: ep.value for ep in entry_points(group="hatch")}
print(f"hatch entry point: {hatch_plugins.get('fancy-pypi-readme')}")
assert hatch_plugins.get("fancy-pypi-readme") == "hatch_fancy_pypi_readme.hooks", hatch_plugins

## Example 2: Text fragments and name/version placeholders

In [ ]:
# Example 2: Run the hook on project metadata with text fragments. $HFPR_PACKAGE_NAME and $HFPR_VERSION
# are replaced with the project's name and version.
import tempfile

from hatch_fancy_pypi_readme.hooks import FancyReadmeMetadataHook

hook_config = {
    "content-type": "text/markdown",
    "fragments": [
        {"text": "# $HFPR_PACKAGE_NAME\n\n"},
        {"text": "Release $HFPR_VERSION of the demo package.\n"},
    ],
}
with tempfile.TemporaryDirectory() as root:
    hook = FancyReadmeMetadataHook(root, hook_config)
    metadata = {"name": "demo-package", "version": "1.2.3"}
    hook.update(metadata)

readme = metadata["readme"]
print(f"content-type: {readme['content-type']}")
print(readme["text"])
assert readme["content-type"] == "text/markdown", readme
assert readme["text"] == "# demo-package\n\nRelease 1.2.3 of the demo package.\n", readme["text"]

## Example 3: File fragments with start/end markers

In [ ]:
# Example 3: File fragments with start-after / end-before markers include only part of a file,
# e.g. the README section between two HTML comments.
import os
import tempfile

from hatch_fancy_pypi_readme._builder import build_text
from hatch_fancy_pypi_readme._config import load_and_validate_config

readme_source = """# Demo project

[![badge](https://example.com/badge.svg)](https://example.com)

<!-- pypi-start -->
Demo project does one thing well.

See the documentation for details.
<!-- pypi-end -->

## Development notes (not for PyPI)
"""

with tempfile.TemporaryDirectory() as tmp:
    readme_path = os.path.join(tmp, "README.md")
    with open(readme_path, "w") as f:
        f.write(readme_source)
    config = load_and_validate_config({
        "content-type": "text/markdown",
        "fragments": [
            {"text": "# Demo project\n"},
            {"path": readme_path, "start-after": "<!-- pypi-start -->\n", "end-before": "<!-- pypi-end -->"},
        ],
    })
    text = build_text(config.fragments, config.substitutions)

print(f"fragments: {[type(f).__name__ for f in config.fragments]}")
print(text)
assert [type(f).__name__ for f in config.fragments] == ["TextFragment", "FileFragment"], config.fragments
assert text == "# Demo project\nDemo project does one thing well.\n\nSee the documentation for details.\n", repr(text)
assert "badge" not in text and "Development notes" not in text

## Example 4: Regex substitutions

In [ ]:
# Example 4: Regex substitutions rewrite the assembled text, e.g. turning relative links into absolute
# URLs (relative links do not resolve on PyPI) and issue references into links.
from hatch_fancy_pypi_readme._builder import build_text
from hatch_fancy_pypi_readme._config import load_and_validate_config

config = load_and_validate_config({
    "content-type": "text/markdown",
    "fragments": [{"text": "See [the changelog](CHANGELOG.md). Fixed in #123 and #45.\n"}],
    "substitutions": [
        {"pattern": r"\]\((?!https?://)(.+?)\)", "replacement": r"](https://example.com/demo/blob/main/\1)"},
        {"pattern": r"#(\d+)", "replacement": r"[#\1](https://example.com/demo/issues/\1)"},
        {"pattern": "CHANGELOG", "replacement": "changes", "ignore-case": True},
    ],
})
text = build_text(config.fragments, config.substitutions)
print(f"substitutions: {len(config.substitutions)}")
print(text)
assert len(config.substitutions) == 3
assert "(https://example.com/demo/blob/main/changes.md)" in text, text
assert "[#123](https://example.com/demo/issues/123)" in text, text
assert "[#45](https://example.com/demo/issues/45)" in text, text

## Example 5: Command-line renderer and configuration errors

In [ ]:
# Example 5: The command-line renderer reads [tool.hatch.metadata.hooks.fancy-pypi-readme] from a
# pyproject.toml and prints the README; invalid configuration is reported as ConfigurationError.
import os
import subprocess
import sys
import tempfile

from hatch_fancy_pypi_readme._config import load_and_validate_config
from hatch_fancy_pypi_readme.exceptions import ConfigurationError

pyproject = """
[project]
name = "demo-package"
dynamic = ["readme"]

[tool.hatch.metadata.hooks.fancy-pypi-readme]
content-type = "text/x-rst"

[[tool.hatch.metadata.hooks.fancy-pypi-readme.fragments]]
text = "Demo package\\n============\\n\\nVersion $HFPR_VERSION.\\n"

[[tool.hatch.metadata.hooks.fancy-pypi-readme.substitutions]]
pattern = "Demo"
replacement = "Sample"
"""

with tempfile.TemporaryDirectory() as tmp:
    pyproject_path = os.path.join(tmp, "pyproject.toml")
    with open(pyproject_path, "w") as f:
        f.write(pyproject)
    result = subprocess.run(
        [sys.executable, "-m", "hatch_fancy_pypi_readme", pyproject_path],
        capture_output=True, text=True, check=True, cwd=tmp,
    )
print(result.stdout)
# The CLI renders with placeholder name "your-package" and version "42.0".
assert result.stdout == "Sample package\n============\n\nVersion 42.0.\n\n", repr(result.stdout)

try:
    load_and_validate_config({"content-type": "text/plain", "fragments": []})
except ConfigurationError as e:
    print("ConfigurationError:")
    for msg in e.errors:
        print(f"  - {msg}")
    assert len(e.errors) == 2, e.errors
    assert any("content-type" in msg for msg in e.errors), e.errors
    assert any("fragments must not be empty" in msg for msg in e.errors), e.errors
else:
    raise AssertionError("expected ConfigurationError for an invalid configuration")